# **Preprocessing**

Notebook ini melakukan:
- transformasi data level menjadi fitur final
- pembuatan target 5-day ahead cumulative return
- pembagian data train / validation / test berdasarkan TARGET_DATE
- pembuatan dataset final untuk ARIMAX, XGBoost, dan LSTM

In [107]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

### **Config**

In [108]:
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_INPUT = PROJECT_ROOT / "data" / "raw" / "gold_macro_raw.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_COL = "DATE"
TARGET_DATE_COL = "TARGET_DATE"
TARGET_COL = "TARGET_5D"
DIRECTION_COL = "TARGET_DIRECTION"

HORIZON = 5

TRAIN_START = "2010-01-01"
TRAIN_END   = "2022-12-31"

VAL_START   = "2023-01-01"
VAL_END     = "2023-12-31"

TEST_START  = "2024-01-01"
TEST_END    = "2025-12-31"

CORE_FEATURES = [
    "GOLD_RET",
    "DXY_RET",
    "SP500_RET",
    "OIL_RET",
    "US10Y_CHANGE",
    "VIX_LEVEL",
]

ARIMAX_EXOG_FEATURES = [
    "DXY_RET",
    "SP500_RET",
    "OIL_RET",
    "US10Y_CHANGE",
    "VIX_LEVEL",
]

XGB_CURRENT_FEATURES = CORE_FEATURES.copy()
XGB_LAG_BASES = CORE_FEATURES.copy()
XGB_LAGS = [1, 2, 3, 4, 5, 6, 7]

LSTM_FEATURES = CORE_FEATURES.copy()
LSTM_WINDOW = 7

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_INPUT   :", DATA_INPUT)
print("OUTPUT_DIR   :", OUTPUT_DIR)

PROJECT_ROOT : C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi
DATA_INPUT   : C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\raw\gold_macro_raw.csv
OUTPUT_DIR   : C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed


### **Load Dataset Raw**

In [109]:
if not DATA_INPUT.exists():
    raise FileNotFoundError(f"Input file not found: {DATA_INPUT}")

df = pd.read_csv(DATA_INPUT)
print("Raw shape:", df.shape)
display(df.head())

Raw shape: (4022, 7)


,DATE,GOLD,DXY,US10Y,VIX,SP500,OIL
0,2010-01-04,1117.699951,77.529999,3.841,20.040001,1132.989990,81.510002
1,2010-01-05,1118.099976,77.620003,3.755,19.350000,1136.520020,81.769997
2,2010-01-06,1135.900024,77.489998,3.808,19.160000,1137.140015,83.180000
3,2010-01-07,1133.099976,77.910004,3.822,19.059999,1141.689941,82.660004
4,2010-01-08,1138.199951,77.470001,3.808,18.129999,1144.979980,82.750000


In [110]:
possible_date_cols = ["DATE", "Date", "date"]
found_date_col = None

for col in possible_date_cols:
    if col in df.columns:
        found_date_col = col
        break

if found_date_col is None:
    raise ValueError(
        f"Date column not found. Expected one of {possible_date_cols}. "
        f"Available columns: {list(df.columns)}"
    )

if found_date_col != DATE_COL:
    df = df.rename(columns={found_date_col: DATE_COL})

required_raw_cols = [DATE_COL, "GOLD", "DXY", "US10Y", "VIX", "SP500", "OIL"]
missing_cols = [c for c in required_raw_cols if c not in df.columns]

if missing_cols:
    raise ValueError(
        f"Missing required raw columns: {missing_cols}\n"
        f"Available columns: {list(df.columns)}"
    )

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

print("Shape after sort:", df.shape)
print("Date range:", df[DATE_COL].min(), "->", df[DATE_COL].max())
print("Duplicate dates:", df[DATE_COL].duplicated().sum())

Shape after sort: (4022, 7)
Date range: 2010-01-04 00:00:00 -> 2025-12-30 00:00:00
Duplicate dates: 0


### **Feature Transformation**

In [111]:
processed_df = df[[DATE_COL, "GOLD", "DXY", "US10Y", "VIX", "SP500", "OIL"]].copy()

processed_df["GOLD_RET"] = processed_df["GOLD"].pct_change()
processed_df["DXY_RET"] = processed_df["DXY"].pct_change()
processed_df["SP500_RET"] = processed_df["SP500"].pct_change()
processed_df["OIL_RET"] = processed_df["OIL"].pct_change()

processed_df["US10Y_CHANGE"] = processed_df["US10Y"].diff()
processed_df["VIX_LEVEL"] = processed_df["VIX"]

processed_df = processed_df.dropna().reset_index(drop=True)

print("Processed shape:", processed_df.shape)
display(processed_df.head())

Processed shape: (4021, 13)


,DATE,GOLD,DXY,US10Y,VIX,SP500,OIL,GOLD_RET,DXY_RET,SP500_RET,OIL_RET,US10Y_CHANGE,VIX_LEVEL
0,2010-01-05,1118.099976,77.620003,3.755,19.350000,1136.520020,81.769997,0.000358,0.001161,0.003116,0.003190,-0.086,19.350000
1,2010-01-06,1135.900024,77.489998,3.808,19.160000,1137.140015,83.180000,0.015920,-0.001675,0.000546,0.017244,0.053,19.160000
2,2010-01-07,1133.099976,77.910004,3.822,19.059999,1141.689941,82.660004,-0.002465,0.005420,0.004001,-0.006251,0.014,19.059999
3,2010-01-08,1138.199951,77.470001,3.808,18.129999,1144.979980,82.750000,0.004501,-0.005648,0.002882,0.001089,-0.014,18.129999
4,2010-01-11,1150.699951,77.000000,3.818,17.549999,1146.979980,82.519997,0.010982,-0.006067,0.001747,-0.002779,0.010,17.549999


In [112]:
processed_output_path = OUTPUT_DIR / "gold_macro_processed_5d.csv"
processed_df.to_csv(processed_output_path, index=False)

print("Saved processed dataset to:", processed_output_path)

Saved processed dataset to: C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\gold_macro_processed_5d.csv


### **Build base dataset and target 5-day ahead cumulative return**

In [113]:
base_df = processed_df[[DATE_COL] + CORE_FEATURES + ["GOLD"]].copy()

base_df[TARGET_DATE_COL] = base_df[DATE_COL].shift(-HORIZON)
base_df[TARGET_COL] = (base_df["GOLD"].shift(-HORIZON) / base_df["GOLD"]) - 1

base_df = base_df.dropna(subset=[TARGET_DATE_COL, TARGET_COL]).reset_index(drop=True)
base_df[TARGET_DATE_COL] = pd.to_datetime(base_df[TARGET_DATE_COL])
base_df[DIRECTION_COL] = (base_df[TARGET_COL] > 0).astype(int)

# kolom GOLD mentah tidak dipakai untuk modeling
base_df = base_df.drop(columns=["GOLD"])

print("Base dataset shape:", base_df.shape)
print("Feature date range:", base_df[DATE_COL].min(), "->", base_df[DATE_COL].max())
print("Target date range :", base_df[TARGET_DATE_COL].min(), "->", base_df[TARGET_DATE_COL].max())

display(base_df.head())
display(base_df.tail())

Base dataset shape: (4016, 10)
Feature date range: 2010-01-05 00:00:00 -> 2025-12-22 00:00:00
Target date range : 2010-01-12 00:00:00 -> 2025-12-30 00:00:00


,DATE,GOLD_RET,DXY_RET,SP500_RET,OIL_RET,US10Y_CHANGE,VIX_LEVEL,TARGET_DATE,TARGET_5D,TARGET_DIRECTION
0,2010-01-05,0.000358,0.001161,0.003116,0.003190,-0.086,19.350000,2010-01-12,0.009659,1
1,2010-01-06,0.015920,-0.001675,0.000546,0.017244,0.053,19.160000,2010-01-13,0.000440,1
2,2010-01-07,-0.002465,0.005420,0.004001,-0.006251,0.014,19.059999,2010-01-14,0.008384,1
3,2010-01-08,0.004501,-0.005648,0.002882,0.001089,-0.014,18.129999,2010-01-15,-0.007116,0
4,2010-01-11,0.010982,-0.006067,0.001747,-0.002779,0.010,17.549999,2010-01-19,-0.009559,0


,DATE,GOLD_RET,DXY_RET,SP500_RET,OIL_RET,US10Y_CHANGE,VIX_LEVEL,TARGET_DATE,TARGET_5D,TARGET_DIRECTION
4011,2025-12-16,-0.000511,0.000000,-0.002384,-0.027279,-0.033,16.480000,2025-12-23,0.041422,1
4012,2025-12-17,0.009990,0.002241,-0.011592,0.012122,0.002,17.620001,2025-12-24,0.030615,1
4013,2025-12-18,-0.001840,0.000610,0.007934,0.003754,-0.035,16.870001,2025-12-26,0.043692,1
4014,2025-12-19,0.005047,0.001727,0.008818,0.009083,0.035,14.910000,2025-12-29,-0.008323,0
4015,2025-12-22,0.019076,-0.003144,0.006436,0.023826,0.018,14.080000,2025-12-30,-0.016762,0


In [114]:
base_output_path = OUTPUT_DIR / "base_dataset_5d.csv"
base_df.to_csv(base_output_path, index=False)

print("Saved base dataset to:", base_output_path)

Saved base dataset to: C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\base_dataset_5d.csv


### **Chronological split**

Data dibagi berdasarkan tanggal baris saat ini:

- train = 2010–2022
- validation = 2023
- test = 2024-2025

Target tetap merepresentasikan next-day gold return dari baris tersebut.

In [115]:
train_base = base_df[
    (base_df[TARGET_DATE_COL] >= pd.Timestamp(TRAIN_START)) &
    (base_df[TARGET_DATE_COL] <= pd.Timestamp(TRAIN_END))
].copy().reset_index(drop=True)

val_base = base_df[
    (base_df[TARGET_DATE_COL] >= pd.Timestamp(VAL_START)) &
    (base_df[TARGET_DATE_COL] <= pd.Timestamp(VAL_END))
].copy().reset_index(drop=True)

test_base = base_df[
    (base_df[TARGET_DATE_COL] >= pd.Timestamp(TEST_START)) &
    (base_df[TARGET_DATE_COL] <= pd.Timestamp(TEST_END))
].copy().reset_index(drop=True)

if train_base.empty or val_base.empty or test_base.empty:
    raise ValueError("One or more base splits are empty.")

print("Train base:", train_base.shape)
print("  feature date:", train_base[DATE_COL].min(), "->", train_base[DATE_COL].max())
print("  target date :", train_base[TARGET_DATE_COL].min(), "->", train_base[TARGET_DATE_COL].max())

print("Val base  :", val_base.shape)
print("  feature date:", val_base[DATE_COL].min(), "->", val_base[DATE_COL].max())
print("  target date :", val_base[TARGET_DATE_COL].min(), "->", val_base[TARGET_DATE_COL].max())

print("Test base :", test_base.shape)
print("  feature date:", test_base[DATE_COL].min(), "->", test_base[DATE_COL].max())
print("  target date :", test_base[TARGET_DATE_COL].min(), "->", test_base[TARGET_DATE_COL].max())

Train base: (3263, 10)
  feature date: 2010-01-05 00:00:00 -> 2022-12-22 00:00:00
  target date : 2010-01-12 00:00:00 -> 2022-12-30 00:00:00
Val base  : (250, 10)
  feature date: 2022-12-23 00:00:00 -> 2023-12-21 00:00:00
  target date : 2023-01-03 00:00:00 -> 2023-12-29 00:00:00
Test base : (503, 10)
  feature date: 2023-12-22 00:00:00 -> 2025-12-22 00:00:00
  target date : 2024-01-02 00:00:00 -> 2025-12-30 00:00:00


In [116]:
train_base_path = OUTPUT_DIR / "train_base_5d.csv"
val_base_path = OUTPUT_DIR / "val_base_5d.csv"
test_base_path = OUTPUT_DIR / "test_base_5d.csv"

train_base.to_csv(train_base_path, index=False)
val_base.to_csv(val_base_path, index=False)
test_base.to_csv(test_base_path, index=False)

print("Saved:")
print("-", train_base_path)
print("-", val_base_path)
print("-", test_base_path)

Saved:
- C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\train_base_5d.csv
- C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\val_base_5d.csv
- C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\test_base_5d.csv


### **Prepare ARIMAX Dataset**

In [117]:
arimax_cols = [
    DATE_COL,
    TARGET_DATE_COL,
    "GOLD_RET",
    TARGET_COL,
    DIRECTION_COL,
] + ARIMAX_EXOG_FEATURES

train_arimax = train_base[arimax_cols].copy()
val_arimax = val_base[arimax_cols].copy()
test_arimax = test_base[arimax_cols].copy()

train_arimax_path = OUTPUT_DIR / "train_arimax_5d.csv"
val_arimax_path = OUTPUT_DIR / "val_arimax_5d.csv"
test_arimax_path = OUTPUT_DIR / "test_arimax_5d.csv"

train_arimax.to_csv(train_arimax_path, index=False)
val_arimax.to_csv(val_arimax_path, index=False)
test_arimax.to_csv(test_arimax_path, index=False)

print("ARIMAX train:", train_arimax.shape)
print("ARIMAX val  :", val_arimax.shape)
print("ARIMAX test :", test_arimax.shape)

ARIMAX train: (3263, 10)
ARIMAX val  : (250, 10)
ARIMAX test : (503, 10)


### **Prepare XGBOOST Dataset**

In [118]:
xgb_df = base_df[
    [DATE_COL, TARGET_DATE_COL] + XGB_CURRENT_FEATURES + [TARGET_COL, DIRECTION_COL]
].copy()

for col in XGB_LAG_BASES:
    for lag in XGB_LAGS:
        xgb_df[f"{col}_LAG{lag}"] = xgb_df[col].shift(lag)

xgb_df = xgb_df.dropna().reset_index(drop=True)

print("XGB full shape:", xgb_df.shape)
display(xgb_df.head())

XGB full shape: (4009, 52)


,DATE,TARGET_DATE,GOLD_RET,DXY_RET,SP500_RET,OIL_RET,US10Y_CHANGE,VIX_LEVEL,TARGET_5D,TARGET_DIRECTION,...,US10Y_CHANGE_LAG5,US10Y_CHANGE_LAG6,US10Y_CHANGE_LAG7,VIX_LEVEL_LAG1,VIX_LEVEL_LAG2,VIX_LEVEL_LAG3,VIX_LEVEL_LAG4,VIX_LEVEL_LAG5,VIX_LEVEL_LAG6,VIX_LEVEL_LAG7
0,2010-01-14,2010-01-22,0.005456,-0.001561,0.002426,-0.003264,-0.049,17.629999,-0.046736,0,...,0.014,0.053,-0.086,17.850000,18.250000,17.549999,18.129999,19.059999,19.160000,19.350000
1,2010-01-15,2010-01-25,-0.010940,0.007689,-0.010823,-0.017508,-0.058,17.910000,-0.030882,0,...,-0.014,0.014,0.053,17.629999,17.850000,18.250000,17.549999,18.129999,19.059999,19.160000
2,2010-01-19,2010-01-26,0.008495,0.002328,0.012500,0.013077,0.031,17.580000,-0.036676,0,...,0.010,-0.014,0.014,17.910000,17.629999,17.850000,18.250000,17.549999,18.129999,19.059999
3,2010-01-20,2010-01-27,-0.024041,0.010839,-0.010598,-0.017717,-0.048,18.680000,-0.025083,0,...,-0.099,0.010,-0.014,17.580000,17.910000,17.629999,17.850000,18.250000,17.549999,18.129999
4,2010-01-21,2010-01-28,-0.008631,-0.000255,-0.018945,-0.019840,-0.048,22.270000,-0.017321,0,...,0.064,-0.099,0.010,18.680000,17.580000,17.910000,17.629999,17.850000,18.250000,17.549999


In [119]:
train_xgb = xgb_df[
    (xgb_df[TARGET_DATE_COL] >= pd.Timestamp(TRAIN_START)) &
    (xgb_df[TARGET_DATE_COL] <= pd.Timestamp(TRAIN_END))
].copy().reset_index(drop=True)

val_xgb = xgb_df[
    (xgb_df[TARGET_DATE_COL] >= pd.Timestamp(VAL_START)) &
    (xgb_df[TARGET_DATE_COL] <= pd.Timestamp(VAL_END))
].copy().reset_index(drop=True)

test_xgb = xgb_df[
    (xgb_df[TARGET_DATE_COL] >= pd.Timestamp(TEST_START)) &
    (xgb_df[TARGET_DATE_COL] <= pd.Timestamp(TEST_END))
].copy().reset_index(drop=True)

if train_xgb.empty or val_xgb.empty or test_xgb.empty:
    raise ValueError("One or more XGBoost splits are empty.")

xgb_feature_cols = [
    c for c in xgb_df.columns
    if c not in [DATE_COL, TARGET_DATE_COL, TARGET_COL, DIRECTION_COL]
]

print("XGB train:", train_xgb.shape)
print("  feature date:", train_xgb[DATE_COL].min(), "->", train_xgb[DATE_COL].max())
print("  target date :", train_xgb[TARGET_DATE_COL].min(), "->", train_xgb[TARGET_DATE_COL].max())

print("XGB val  :", val_xgb.shape)
print("  feature date:", val_xgb[DATE_COL].min(), "->", val_xgb[DATE_COL].max())
print("  target date :", val_xgb[TARGET_DATE_COL].min(), "->", val_xgb[TARGET_DATE_COL].max())

print("XGB test :", test_xgb.shape)
print("  feature date:", test_xgb[DATE_COL].min(), "->", test_xgb[DATE_COL].max())
print("  target date :", test_xgb[TARGET_DATE_COL].min(), "->", test_xgb[TARGET_DATE_COL].max())

print("XGB number of features:", len(xgb_feature_cols))

XGB train: (3256, 52)
  feature date: 2010-01-14 00:00:00 -> 2022-12-22 00:00:00
  target date : 2010-01-22 00:00:00 -> 2022-12-30 00:00:00
XGB val  : (250, 52)
  feature date: 2022-12-23 00:00:00 -> 2023-12-21 00:00:00
  target date : 2023-01-03 00:00:00 -> 2023-12-29 00:00:00
XGB test : (503, 52)
  feature date: 2023-12-22 00:00:00 -> 2025-12-22 00:00:00
  target date : 2024-01-02 00:00:00 -> 2025-12-30 00:00:00
XGB number of features: 48


In [120]:
train_xgb_path = OUTPUT_DIR / "train_xgb_5d.csv"
val_xgb_path = OUTPUT_DIR / "val_xgb_5d.csv"
test_xgb_path = OUTPUT_DIR / "test_xgb_5d.csv"

train_xgb.to_csv(train_xgb_path, index=False)
val_xgb.to_csv(val_xgb_path, index=False)
test_xgb.to_csv(test_xgb_path, index=False)

print("Saved:")
print("-", train_xgb_path)
print("-", val_xgb_path)
print("-", test_xgb_path)

Saved:
- C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\train_xgb_5d.csv
- C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\val_xgb_5d.csv
- C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\test_xgb_5d.csv


### **Prepare LSTM Dataset**

In [121]:
lstm_source_df = base_df[[DATE_COL, TARGET_DATE_COL] + LSTM_FEATURES + [TARGET_COL, DIRECTION_COL]].copy()

sequence_rows = []

for end_idx in range(LSTM_WINDOW - 1, len(lstm_source_df)):
    row = {
        DATE_COL: lstm_source_df.loc[end_idx, DATE_COL],
        TARGET_DATE_COL: lstm_source_df.loc[end_idx, TARGET_DATE_COL],
        TARGET_COL: lstm_source_df.loc[end_idx, TARGET_COL],
        DIRECTION_COL: lstm_source_df.loc[end_idx, DIRECTION_COL],
    }

    start_idx = end_idx - LSTM_WINDOW + 1

    for step, src_idx in enumerate(range(start_idx, end_idx + 1)):
        for feature in LSTM_FEATURES:
            row[f"{feature}_T{step}"] = lstm_source_df.loc[src_idx, feature]

    sequence_rows.append(row)

lstm_df = pd.DataFrame(sequence_rows)

print("LSTM full sequence dataset shape:", lstm_df.shape)
display(lstm_df.head())

LSTM full sequence dataset shape: (4010, 46)


,DATE,TARGET_DATE,TARGET_5D,TARGET_DIRECTION,GOLD_RET_T0,DXY_RET_T0,SP500_RET_T0,OIL_RET_T0,US10Y_CHANGE_T0,VIX_LEVEL_T0,...,SP500_RET_T5,OIL_RET_T5,US10Y_CHANGE_T5,VIX_LEVEL_T5,GOLD_RET_T6,DXY_RET_T6,SP500_RET_T6,OIL_RET_T6,US10Y_CHANGE_T6,VIX_LEVEL_T6
0,2010-01-13,2010-01-21,-0.029655,0,0.000358,0.001161,0.003116,0.003190,-0.086,19.350000,...,-0.009381,-0.020965,-0.099,18.250000,0.006644,-0.001300,0.008326,-0.014111,0.064,17.850000
1,2010-01-14,2010-01-22,-0.046736,0,0.015920,-0.001675,0.000546,0.017244,0.053,19.160000,...,0.008326,-0.014111,0.064,17.850000,0.005456,-0.001561,0.002426,-0.003264,-0.049,17.629999
2,2010-01-15,2010-01-25,-0.030882,0,-0.002465,0.005420,0.004001,-0.006251,0.014,19.059999,...,0.002426,-0.003264,-0.049,17.629999,-0.010940,0.007689,-0.010823,-0.017508,-0.058,17.910000
3,2010-01-19,2010-01-26,-0.036676,0,0.004501,-0.005648,0.002882,0.001089,-0.014,18.129999,...,-0.010823,-0.017508,-0.058,17.910000,0.008495,0.002328,0.012500,0.013077,0.031,17.580000
4,2010-01-20,2010-01-27,-0.025083,0,0.010982,-0.006067,0.001747,-0.002779,0.010,17.549999,...,0.012500,0.013077,0.031,17.580000,-0.024041,0.010839,-0.010598,-0.017717,-0.048,18.680000


In [122]:
train_lstm = lstm_df[
    (lstm_df[TARGET_DATE_COL] >= pd.Timestamp(TRAIN_START)) &
    (lstm_df[TARGET_DATE_COL] <= pd.Timestamp(TRAIN_END))
].copy().reset_index(drop=True)

val_lstm = lstm_df[
    (lstm_df[TARGET_DATE_COL] >= pd.Timestamp(VAL_START)) &
    (lstm_df[TARGET_DATE_COL] <= pd.Timestamp(VAL_END))
].copy().reset_index(drop=True)

test_lstm = lstm_df[
    (lstm_df[TARGET_DATE_COL] >= pd.Timestamp(TEST_START)) &
    (lstm_df[TARGET_DATE_COL] <= pd.Timestamp(TEST_END))
].copy().reset_index(drop=True)

if train_lstm.empty or val_lstm.empty or test_lstm.empty:
    raise ValueError("One or more LSTM splits are empty.")

train_lstm_path = OUTPUT_DIR / "train_lstm_5d.csv"
val_lstm_path = OUTPUT_DIR / "val_lstm_5d.csv"
test_lstm_path = OUTPUT_DIR / "test_lstm_5d.csv"

train_lstm.to_csv(train_lstm_path, index=False)
val_lstm.to_csv(val_lstm_path, index=False)
test_lstm.to_csv(test_lstm_path, index=False)

print("LSTM train:", train_lstm.shape)
print("  target date :", train_lstm[TARGET_DATE_COL].min(), "->", train_lstm[TARGET_DATE_COL].max())

print("LSTM val  :", val_lstm.shape)
print("  target date :", val_lstm[TARGET_DATE_COL].min(), "->", val_lstm[TARGET_DATE_COL].max())

print("LSTM test :", test_lstm.shape)
print("  target date :", test_lstm[TARGET_DATE_COL].min(), "->", test_lstm[TARGET_DATE_COL].max())

LSTM train: (3257, 46)
  target date : 2010-01-21 00:00:00 -> 2022-12-30 00:00:00
LSTM val  : (250, 46)
  target date : 2023-01-03 00:00:00 -> 2023-12-29 00:00:00
LSTM test : (503, 46)
  target date : 2024-01-02 00:00:00 -> 2025-12-30 00:00:00


### **Sanity checks**

Bagian ini memastikan:
- tidak ada missing value
- split sudah benar
- file output sudah siap dipakai notebook model

In [123]:
datasets_to_check = {
    "processed_df": processed_df,
    "base_df": base_df,
    "train_base": train_base,
    "val_base": val_base,
    "test_base": test_base,
    "train_arimax": train_arimax,
    "val_arimax": val_arimax,
    "test_arimax": test_arimax,
    "train_xgb": train_xgb,
    "val_xgb": val_xgb,
    "test_xgb": test_xgb,
    "lstm_df": lstm_df,
    "train_lstm": train_lstm,
    "val_lstm": val_lstm,
    "test_lstm": test_lstm,
}

for name, data in datasets_to_check.items():
    missing = data.isna().sum().sum()
    print(f"{name:15s} | shape={data.shape} | missing={missing}")
    if missing > 0:
        raise ValueError(f"{name} still contains missing values.")

print("\nAll sanity checks passed.")

processed_df    | shape=(4021, 13) | missing=0
base_df         | shape=(4016, 10) | missing=0
train_base      | shape=(3263, 10) | missing=0
val_base        | shape=(250, 10) | missing=0
test_base       | shape=(503, 10) | missing=0
train_arimax    | shape=(3263, 10) | missing=0
val_arimax      | shape=(250, 10) | missing=0
test_arimax     | shape=(503, 10) | missing=0
train_xgb       | shape=(3256, 52) | missing=0
val_xgb         | shape=(250, 52) | missing=0
test_xgb        | shape=(503, 52) | missing=0
lstm_df         | shape=(4010, 46) | missing=0
train_lstm      | shape=(3257, 46) | missing=0
val_lstm        | shape=(250, 46) | missing=0
test_lstm       | shape=(503, 46) | missing=0

All sanity checks passed.


### **Save Metadata**

In [124]:
metadata = {
    "input_file": str(DATA_INPUT),
    "output_dir": str(OUTPUT_DIR),
    "horizon_days": HORIZON,
    "feature_policy": "core features only",
    "target_definition": "TARGET_5D = (GOLD_t+5 / GOLD_t) - 1",
    "target_date_definition": "TARGET_DATE = DATE.shift(-5)",
    "direction_definition": "TARGET_DIRECTION = 1 if TARGET_5D > 0 else 0",
    "split_policy": "split by TARGET_DATE to prevent label leakage across train/validation/test boundary",
    "core_features": CORE_FEATURES,
    "arimax_exog_features": ARIMAX_EXOG_FEATURES,
    "xgb_current_features": XGB_CURRENT_FEATURES,
    "xgb_lag_bases": XGB_LAG_BASES,
    "xgb_lags": XGB_LAGS,
    "lstm_features": LSTM_FEATURES,
    "lstm_window": LSTM_WINDOW,
    "lstm_sequence_policy": "sequence built globally first, then split by TARGET_DATE",
    "split_definition": {
        "train": f"TARGET_DATE {TRAIN_START} to {TRAIN_END}",
        "validation": f"TARGET_DATE {VAL_START} to {VAL_END}",
        "test": f"TARGET_DATE {TEST_START} to {TEST_END}",
    },
    "output_files": {
        "processed_dataset": str(processed_output_path),
        "base_dataset": str(base_output_path),
        "train_base": str(train_base_path),
        "val_base": str(val_base_path),
        "test_base": str(test_base_path),
        "train_arimax": str(train_arimax_path),
        "val_arimax": str(val_arimax_path),
        "test_arimax": str(test_arimax_path),
        "train_xgb": str(train_xgb_path),
        "val_xgb": str(val_xgb_path),
        "test_xgb": str(test_xgb_path),
        "train_lstm": str(train_lstm_path),
        "val_lstm": str(val_lstm_path),
        "test_lstm": str(test_lstm_path),
    },
    "row_counts": {
        "processed_df": len(processed_df),
        "base_df": len(base_df),
        "train_base": len(train_base),
        "val_base": len(val_base),
        "test_base": len(test_base),
        "train_arimax": len(train_arimax),
        "val_arimax": len(val_arimax),
        "test_arimax": len(test_arimax),
        "train_xgb": len(train_xgb),
        "val_xgb": len(val_xgb),
        "test_xgb": len(test_xgb),
        "lstm_df": len(lstm_df),
        "train_lstm": len(train_lstm),
        "val_lstm": len(val_lstm),
        "test_lstm": len(test_lstm),
    },
}

metadata_path = OUTPUT_DIR / "feature_metadata_5d.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata to:", metadata_path)

Saved metadata to: C:\Users\Afzi\Documents\BINUS\0. SKRIPSI\gold_forecasting_skripsi\data\processed\feature_metadata_5d.json


### **Final Summary**

In [126]:
print("\n===== FINAL SUMMARY =====")
print("Processed dataset :", processed_df.shape)
print("Base dataset      :", base_df.shape)
print("Train base        :", train_base.shape)
print("Val base          :", val_base.shape)
print("Test base         :", test_base.shape)
print("Train ARIMAX      :", train_arimax.shape)
print("Val ARIMAX        :", val_arimax.shape)
print("Test ARIMAX       :", test_arimax.shape)
print("Train XGBoost     :", train_xgb.shape)
print("Val XGBoost       :", val_xgb.shape)
print("Test XGBoost      :", test_xgb.shape)
print("LSTM full seq     :", lstm_df.shape)
print("Train LSTM        :", train_lstm.shape)
print("Val LSTM          :", val_lstm.shape)
print("Test LSTM         :", test_lstm.shape)

print("\nSaved files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)


===== FINAL SUMMARY =====
Processed dataset : (4021, 13)
Base dataset      : (4016, 10)
Train base        : (3263, 10)
Val base          : (250, 10)
Test base         : (503, 10)
Train ARIMAX      : (3263, 10)
Val ARIMAX        : (250, 10)
Test ARIMAX       : (503, 10)
Train XGBoost     : (3256, 52)
Val XGBoost       : (250, 52)
Test XGBoost      : (503, 52)
LSTM full seq     : (4010, 46)
Train LSTM        : (3257, 46)
Val LSTM          : (250, 46)
Test LSTM         : (503, 46)

Saved files:
- archive
- base_dataset_5d.csv
- feature_engineering
- feature_metadata_5d.json
- gold_macro_processed_5d.csv
- test_arimax_5d.csv
- test_base_5d.csv
- test_lstm_5d.csv
- test_xgb_5d.csv
- train_arimax_5d.csv
- train_base_5d.csv
- train_lstm_5d.csv
- train_xgb_5d.csv
- val_arimax_5d.csv
- val_base_5d.csv
- val_lstm_5d.csv
- val_xgb_5d.csv
